# ApproxHDC-Forge: Colab flow test

Checks that the RTL-to-GDS flow runs in Google Colab **without Docker**. It installs LibreLane's tools with Nix (cells 1–2 are copied from LibreLane's official notebook, Apache-2.0, pinned to LibreLane 3.0.14), then pushes one small ApproxHDC design (D=256, P=32, learning off) through LibreLane on sky130A.

Colab has only 2 vCPUs, so only this small design runs live. All other flow results in the main notebook are replayed from `results/flow_runs.csv`. Expect the setup to take about 10 minutes and the flow about 30–60 minutes.

In [ ]:
# @title Setup Nix {display-mode: "form"}
# @markdown <img src="https://raw.githubusercontent.com/NixOS/nixos-artwork/51a27e4a011e95cb559e37d32c44cf89b50f5154/logo/nix-snowflake-colours.svg" width="32"/>
# @markdown
# @markdown Nix is a package manager with an emphasis on reproducible builds,
# @markdown and it is the primary method for installing LibreLane.
# @markdown
# @markdown This step installs the Nix package manager and enables the
# @markdown FOSSi Foundation Nix Cache.
# @markdown
# @markdown If you're not in a Colab, this just sets the environment variables.
# @markdown You will need to install Nix and enable flakes on your own following
# @markdown [this guide](https://librelane.readthedocs.io/en/stable/getting_started/common/nix_installation/index.html).
import os
from pathlib import Path
import subprocess
import sys
import shutil
import tempfile

os.environ["LOCALE_ARCHIVE"] = "/usr/lib/locale/locale-archive"

if "google.colab" in sys.modules:
    if shutil.which("nix-env") is None:
        with tempfile.TemporaryDirectory() as d:
            d = Path(d)
            installer_path = d / "nix"
            !curl --proto '=https' --tlsv1.2 -sSf -L https://install.determinate.systems/nix > {installer_path}
            with subprocess.Popen(
                [
                    "bash",
                    installer_path,
                    "install",
                    "--prefer-upstream-nix",
                    "--no-confirm",
                    "--extra-conf",
                    "extra-substituters = https://nix-cache.fossi-foundation.org\nextra-trusted-public-keys = nix-cache.fossi-foundation.org:3+K59iFwXqKsL7BNu6Guy0v+uTlwsxYQxjspXzqLYQs=\n",
                ],
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                encoding="utf8",
            ) as p:
                for line in p.stdout:
                    print(line, end="")
else:
    if shutil.which("nix-env") is None:
        raise RuntimeError("Nix is not installed!")

os.environ["PATH"] = f"/nix/var/nix/profiles/default/bin/:{os.getenv('PATH')}"

In [ ]:
# @title Get LibreLane {display-mode: "form"}
# @markdown Click the ▷ button to download and install LibreLane.
# @markdown
# @markdown This will install LibreLane's tool dependencies using Nix,
# @markdown and LibreLane itself using PIP.
# @markdown
# @markdown Note that `python3-tk` may need to be installed using your OS's
# @markdown package manager.
import os
import yaml
import subprocess
import IPython

librelane_version = "3.0.14"  # @param {key:"LibreLane Version", type:"string"}

if librelane_version == "latest":
    librelane_version = "main"

pdk_root = "~/.ciel"  # @param {key:"PDK Root", type:"string"}

pdk_root = os.path.expanduser(pdk_root)

pdk = "sky130"  # @param {key:"PDK (without the variant)", type:"string"}

librelane_ipynb_path = os.path.join(os.getcwd(), "librelane_ipynb")

display(IPython.display.HTML("<h3>Downloading LibreLane…</a>"))


TESTING_LOCALLY = False
!rm -rf {librelane_ipynb_path}
!mkdir -p {librelane_ipynb_path}
if TESTING_LOCALLY:
    !ln -s {os.getcwd()} {librelane_ipynb_path}
else:
    !curl -L "https://github.com/librelane/librelane/tarball/{librelane_version}" | tar -xzC {librelane_ipynb_path} --strip-components 1

try:
    import tkinter
except ImportError:
    if "google.colab" in sys.modules:
        !sudo apt-get install python-tk

try:
    import tkinter
except ImportError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to import the <code>tkinter</code> library for Python, which is required to load PDK configuration values. Make sure <code>python3-tk</code> or equivalent is installed on your system.</a>'
        )
    )
    raise e from None


display(IPython.display.HTML("<h3>Downloading LibreLane's dependencies…</a>"))
try:
    with subprocess.Popen(
        [
            "nix",
            "profile",
            "install",
            ".#colab-env",
        ],
        cwd=librelane_ipynb_path,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        encoding="utf8",
    ) as p:
        for line in p.stdout:
            print(line, end="")
except subprocess.CalledProcessError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to install binary dependencies using Nix…</h3>'
        )
    )

display(IPython.display.HTML("<h3>Downloading Python dependencies using PIP…</a>"))
try:
    subprocess.check_call(
        ["pip3", "install", "."],
        cwd=librelane_ipynb_path,
    )
except subprocess.CalledProcessError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to install Python dependencies using PIP…</h3>'
        )
    )
    raise e from None

display(IPython.display.HTML("<h3>Downloading PDK…</a>"))
import ciel
from ciel.source import StaticWebDataSource

with open(
    os.path.join(librelane_ipynb_path, "librelane", "pdk_hashes.yaml"), "r"
) as file:
    pdk_hashes = yaml.safe_load(file)

ciel.enable(
    ciel.get_ciel_home(pdk_root),
    pdk,
    pdk_hashes[pdk],
    data_source=StaticWebDataSource("https://fossi-foundation.github.io/ciel-releases"),
)

sys.path.insert(0, librelane_ipynb_path)
display(IPython.display.HTML("<h3>⭕️ Done.</a>"))

import logging

# Remove the stupid default colab logging handler
logging.getLogger().handlers.clear()

In [ ]:
# @title Get ApproxHDC-Forge (sparse clone: only this folder, not the whole repo)
REPO = "https://github.com/Mohit14036/sscs-ose-code-a-chip.github.io"  # @param {type:"string"}
BRANCH = "approxhdc-forge"  # @param {type:"string"}
SUB = "ISSCC27/submitted_notebooks/approxhdc_forge"
!rm -rf hdc_repo && git clone -q --depth 1 --filter=blob:none --sparse -b {BRANCH} {REPO} hdc_repo && cd hdc_repo && git sparse-checkout set {SUB}
%cd hdc_repo/{SUB}
!pip install -q numpy scipy pandas scikit-learn matplotlib pyyaml
import sys; sys.path.insert(0, "src")

In [ ]:
# @title Run one small design through LibreLane (no Docker)
import os, time, json, importlib
os.environ['HDC_FLOW_NATIVE'] = '1'   # no Docker: flow_runner wraps the Nix yosys (see _native_yosys_wrapper)
os.environ['PDK_ROOT'] = os.path.expanduser('~/.ciel')
import flow_runner as F
importlib.reload(F)   # pick up a re-cloned flow_runner.py without restarting the runtime
from rtlgen import RTLConfig
F.CSV_PATH = F.ROOT / 'results' / 'flow_runs_colab.csv'   # keep the replayed CSV untouched
cfg = RTLConfig(D=256, P=32, learn=0)
t = time.time()
row = F.run(cfg, clock_ns=20.0, threads=2, render=True, mode='fast', tag='colab')
print(f'wall time {(time.time() - t) / 60:.1f} min')
print(json.dumps({k: row[k] for k in ('flow_ok', 'signoff_clean', 'cell_area_um2', 'core_area_um2', 'setup_ws_tt_ns', 'fmax_tt_mhz', 'power_total_w', 'klayout_drc', 'lvs_errors', 'route_drc')}, indent=2))
if not row['flow_ok']:
    run_dir = F.design_dir(cfg, 20.0) / 'runs' / 'colab'
    failed = sorted(p for p in run_dir.iterdir() if p.is_dir() and p.name[:2].isdigit())[-1]
    print('last step:', failed.name)
    for log in failed.glob('*.log'):
        print(f'--- {log.name} (last 40 lines)')
        print(''.join(open(log).readlines()[-40:]))

In [ ]:
# @title Show the layout and compare with the local (Docker) run of the same design
import pandas as pd
from IPython.display import Image, display
png = F.FIGS / f'layout_{cfg.name}.png'
if png.exists():
    display(Image(str(png), width=600))
else:
    print('(the layout image could not be rendered in this environment; the flow results below are unaffected)')
local = pd.read_csv('results/flow_runs.csv')
m = local[(local.D == 256) & (local.P == 32) & (local.learn == 0) & (local.variant == 'exact')]
cols = ['cell_area_um2', 'core_area_um2', 'setup_ws_tt_ns', 'power_total_w', 'signoff_clean']
print('local Docker run:'); print(m[cols].to_string(index=False) if len(m) else '  (not in CSV yet)')
print('Colab run:'); print(pd.DataFrame([row])[cols].to_string(index=False))
if len(m) and row['flow_ok']:
    a, b = float(m.cell_area_um2.iloc[0]), float(row['cell_area_um2'])
    print(f"\nstd-cell area differs by {abs(a - b) / a:.3%} between Docker and Colab; "
          f"sign-off clean: Docker {int(m.signoff_clean.iloc[0])}, Colab {int(row['signoff_clean'])}")